In [2]:

#Bước 1: Tự tính khoảng cách và tự tìm láng giềng, chưa dùng thư viện.

import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f" d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f" d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hóa bằng trung bình và độ lệch chuẩn của cả bộ dữ liệu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/sinh_vien_mo_rong.csv")

X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)

print("Do lech chuan cua ba cot:")
print(
    f" gio_on {dl[0]:.2f} diem_giua_ky {dl[1]:.2f}"
    f" chi_tieu {dl[2]:.0f}"
)
print()

chuan = {
    ten: (v - tb) / dl
    for ten, v in ba_ban.items()
}

print("Khoang cach SAU KHI chuan hoa")
print(f" d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f" d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉ dùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()

tb2 = X2.mean(axis=0)
dl2 = X2.std(axis=0)

ban_moi = np.array([12.0, 6.5])

X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2

khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")

for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(
        f" gio {X2[i, 0]:5.1f} diem {X2[i, 1]:4.1f} "
        f"khoang cach {khoang[i]:.3f} -> {ket}"
    )

so_qua = int(y[gan].sum())

print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")


Khoang cach KHI CHUA chuan hoa
 d(A, B) =    102.2
 d(A, C) =   3000.0

Do lech chuan cua ba cot:
 gio_on 7.82 diem_giua_ky 2.09 chi_tieu 1692

Khoang cach SAU KHI chuan hoa
 d(A, B) = 3.85
 d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
 gio  10.5 diem  6.4 khoang cach 0.198 -> qua
 gio  12.5 diem  6.1 khoang cach 0.202 -> qua
 gio  14.0 diem  6.4 khoang cach 0.260 -> qua
 gio   9.5 diem  6.1 khoang cach 0.373 -> rot
 gio  10.5 diem  5.8 khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


In [3]:


#Bước 2: Đo hậu quả của việc quên chuẩn hóa trên tập kiểm tra.

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Đọc dữ liệu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/sinh_vien_mo_rong.csv")

y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X3,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)


def cham(mo_hinh, cot):
    """Danh gia do chinh xac cua mo hinh."""
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=7)
    )


# Các cột đặc trưng được sử dụng
ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

# So sánh ba trường hợp
a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

# In kết quả
print(f"Doan bua ca lop qua : {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa : {a:.4f}")
print(f"Ba cot, co chuan hoa : {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa: {c:.4f}")


Doan bua ca lop qua : 0.6000
Ba cot, khong chuan hoa : 0.5333
Ba cot, co chuan hoa : 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


In [4]:


#Bước 3: Chọn số láng giềng k bằng kiểm định chéo.

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

# Đọc dữ liệu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia tập huấn luyện và tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉ chênh khoảng
# 1.6 lần, nên ở bộ này không chuẩn hóa cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print(" k   tap hoc   tap kiem tra")

for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {k:3d}   {a_hoc:.4f}       {a_kt:.4f}")

print()

# Chọn k chỉ dựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")

diem = {}

for k in range(1, 52, 2):
    diem[k] = cross_val_score(
        KNeighborsClassifier(n_neighbors=k),
        X_tr,
        y_tr,
        cv=5
    ).mean()

k_tot = max(diem, key=diem.get)

for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = " <- tot nhat" if k == k_tot else ""
    print(f" k = {k:2d} diem trung binh {diem[k]:.4f}{dau}")

print()

# Chọn xong mới đánh giá trên tập kiểm tra
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)

print(f"k tot nhat = {k_tot}")

acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")


Accuracy tren tap hoc va tap kiem tra theo k
 k   tap hoc   tap kiem tra
   1   1.0000       0.8000
   5   0.8756       0.9067
  13   0.8844       0.9200
  31   0.8667       0.8933
 101   0.8311       0.8533

Kiem dinh cheo 5 phan tren tap hoc:
 k =  1 diem trung binh 0.7778
 k =  5 diem trung binh 0.8444
 k =  9 diem trung binh 0.8444
 k = 13 diem trung binh 0.8667 <- tot nhat
 k = 17 diem trung binh 0.8533
 k = 25 diem trung binh 0.8622
 k = 51 diem trung binh 0.8533

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


In [5]:

#Bước 4: Đặt KNN cạnh hai mô hình đã học trên dữ liệu có biên cong.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Đọc dữ liệu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/hai_vang_trang.csv")

X = df[["x1", "x2"]]
y = df["nhan"]

# Chia dữ liệu thành tập huấn luyện và tập kiểm tra
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=17,
    stratify=y
)

# Khởi tạo các mô hình
cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes", GaussianNB()),
    ("KNN, k = 13", KNeighborsClassifier(n_neighbors=13)),
]

# Huấn luyện và đánh giá các mô hình
print("Accuracy tren tap kiem tra, bo hai vang trang")

for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    y_du_doan = mh.predict(X_te)
    acc = accuracy_score(y_te, y_du_doan)
    print(f" {ten}: {acc:.4f}")


Accuracy tren tap kiem tra, bo hai vang trang
 Hoi quy logistic: 0.8533
 Naive Bayes: 0.8533
 KNN, k = 13: 0.9200


In [ ]:

# -*- coding: utf-8 -*-
"""
Bước 5: KNN cho bài toán hồi quy, trên dữ liệu giá nhà của bài 1.
"""

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

# Đọc dữ liệu
df = pd.read_csv("/Users/buixuankhanh/Documents/machine-learning/04_KNN/data/gia_nha.csv")

X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 để so sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Huấn luyện hai mô hình
knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

# So sánh MAE trên tập kiểm tra
print("MAE tren tap kiem tra (ty dong)")

mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))

print(f" KNN, k = 5 : {mae_knn:.4f}")
print(f" Duong thang : {mae_duong:.4f}")
print()

# Dự đoán giá cho bốn căn nhà mới
can_moi = pd.DataFrame({
    "dien_tich": [60.0, 100.0, 150.0, 250.0]
})

du_doan_knn = knn.predict(can_moi)
du_doan_duong = duong.predict(can_moi)

print("Du doan cho bon can moi (ty dong)")
print(" dien tich    KNN    duong thang")

for dt, a, b in zip(
    can_moi["dien_tich"],
    du_doan_knn,
    du_doan_duong
):
    print(f" {dt:6.0f} m2  {a:6.2f}  {b:6.2f}")

print()

# Tìm năm láng giềng gần nhất của căn nhà 250 m2
_, chi_so = knn.kneighbors(
    pd.DataFrame({"dien_tich": [250.0]})
)

lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()

print(
    "Nam lang gieng cua can 250 m2:",
    ", ".join(f"{v:.1f}" for v in lg)
)
